# 2.2. Задача трёх тел

Исследуем плоское движение трёх материальных точек, взаимно притягивающихся по закону Ньютона. Сравним метод Рунге — Кутты четвёртого порядка (RK4) и метод Верлета для трёх случаев: иерархической системы из звезды и двух лёгких планет, периодической «восьмёрки» равных масс и пролёта третьего тела около связанной пары. По изменению траекторий и интегралов движения при уменьшении шага определим, какие участки требуют наиболее мелкой сетки.

Используем безразмерные величины. При единицах длины $\ell$ и массы $M$ единица времени равна $t_*=\sqrt{\ell^3/(GM)}$; безразмерная гравитационная постоянная тогда равна 1. Все три тела подвижны. Потенциал ньютоновский, без сглаживания; минимальные расстояния в выбранных движениях остаются ненулевыми.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.lines import Line2D
from IPython.display import HTML, Markdown, display

plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.16,
    'axes.axisbelow': True,
    'lines.linewidth': 1.7,
    'legend.framealpha': 1.0,
    'legend.handlelength': 3.0,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'animation.embed_limit': 80,
    'animation.frame_format': 'jpeg',
})

G = 1.0
BODY_COLORS = ['#0072B2', '#D55E00', '#009E73']
BODY_MARKERS = ['o', 's', '^']
METHOD_COLORS = {'RK4': '#0072B2', 'Метод Верлета': '#D55E00'}
METHOD_LINES = {'RK4': '-', 'Метод Верлета': (0, (6, 3))}
METHOD_MARKERS = {'RK4': 'o', 'Метод Верлета': 's'}
PAIRS = [(0, 1), (0, 2), (1, 2)]


def table(headers, rows):
    def line(items):
        return '| ' + ' | '.join(str(x).replace('|', '&#124;') for x in items) + ' |'
    display(Markdown('\n'.join([
        line(headers), line(['---'] * len(headers)), *[line(row) for row in rows]
    ])))

## 1. Уравнения и численные схемы

Для тела $i$ с массой $m_i$, положением $\mathbf r_i=(x_i,y_i)$ и скоростью $\mathbf v_i$ уравнения движения имеют вид
$$
\dot{\mathbf r}_i=\mathbf v_i,\qquad
\dot{\mathbf v}_i=\mathbf a_i(\mathbf r)
=G\sum_{j\ne i}m_j\frac{\mathbf r_j-\mathbf r_i}{|\mathbf r_j-\mathbf r_i|^3}.
$$
Собственная масса сокращается при делении силы на $m_i$; в ускорении остаётся масса притягивающего тела $m_j$. Для трёх тел получаем систему из двенадцати уравнений первого порядка: по две координаты и две компоненты скорости на каждое тело.

В функции `acceleration` массив `dr[i, j]` содержит вектор $\mathbf r_j-\mathbf r_i$. Бесконечные значения на диагонали массива квадратов расстояний исключают члены с $i=j$ из суммы. Массивы координат и скоростей имеют размер `(3, 2)`.

In [ ]:
def acceleration(r, masses):
    dr = r[None, :, :] - r[:, None, :]
    distance2 = np.sum(dr**2, axis=2)
    np.fill_diagonal(distance2, np.inf)
    return G * np.sum(
        masses[None, :, None] * dr / distance2[:, :, None]**1.5, axis=1
    )


def to_center_of_mass(r, v, masses):
    return (r - np.average(r, weights=masses, axis=0),
            v - np.average(v, weights=masses, axis=0))

Для RK4 объединяем координаты и скорости в состояние $s=(\mathbf r,\mathbf v)$ с правой частью $F(s)=(\mathbf v,\mathbf a(\mathbf r))$. Один шаг:
$$
\begin{aligned}
k_1&=F(s_n),&k_2&=F(s_n+hk_1/2),\\
k_3&=F(s_n+hk_2/2),&k_4&=F(s_n+hk_3),\\
s_{n+1}&=s_n+\frac h6(k_1+2k_2+2k_3+k_4).
\end{aligned}
$$

Скоростная форма **метода Верлета** состоит из двух обновлений скорости на полшага и одного обновления координат:
$$
\begin{aligned}
\mathbf v_{n+1/2}&=\mathbf v_n+\tfrac h2\mathbf a(\mathbf r_n),\\
\mathbf r_{n+1}&=\mathbf r_n+h\mathbf v_{n+1/2},\\
\mathbf v_{n+1}&=\mathbf v_{n+1/2}+\tfrac h2\mathbf a(\mathbf r_{n+1}).
\end{aligned}
$$
Подставив промежуточную скорость, получаем эквивалентные формулы
$$
\mathbf r_{n+1}=\mathbf r_n+h\mathbf v_n+\frac{h^2}{2}\mathbf a_n,\qquad
\mathbf v_{n+1}=\mathbf v_n+\frac h2(\mathbf a_n+\mathbf a_{n+1}).
$$
Ускорение после перемещения пересчитывается по новым координатам всех тел. Эта схема применима к рассматриваемой задаче, поскольку сила не зависит от скорости.

На фиксированном промежутке времени глобальная погрешность имеет порядок $O(h^4)$ для RK4 и $O(h^2)$ для метода Верлета при достаточно малом шаге. Задаём целое число шагов $N$ и $h=t_{\mathrm{end}}/N$. При удвоении $N$ сравниваем решения в общих узлах вложенных сеток; интерполяция не требуется.

In [ ]:
def rk4_step(r, v, h, masses):
    k1r = v
    k1v = acceleration(r, masses)
    k2r = v + h*k1v/2
    k2v = acceleration(r + h*k1r/2, masses)
    k3r = v + h*k2v/2
    k3v = acceleration(r + h*k2r/2, masses)
    k4r = v + h*k3v
    k4v = acceleration(r + h*k3r, masses)
    return (r + h*(k1r + 2*k2r + 2*k3r + k4r)/6,
            v + h*(k1v + 2*k2v + 2*k3v + k4v)/6)


def verlet_step(r, v, h, masses):
    v_half = v + h/2*acceleration(r, masses)
    r_next = r + h*v_half
    v_next = v_half + h/2*acceleration(r_next, masses)
    return r_next, v_next


METHODS = {'RK4': rk4_step, 'Метод Верлета': verlet_step}


def integrate(r0, v0, masses, t_end, n_steps, method):
    t = np.linspace(0, t_end, n_steps + 1)
    h = t_end/n_steps
    r = np.empty((n_steps + 1, len(masses), 2))
    v = np.empty_like(r)
    r[0], v[0] = r0, v0
    step = METHODS[method]
    for n in range(n_steps):
        r[n+1], v[n+1] = step(r[n], v[n], h, masses)
    return {'t': t, 'r': r, 'v': v, 'h': h, 'N': n_steps}

### Проверка на круговом движении двух тел

Возьмём $m_1=1$, $m_2=0.2$ и постоянное расстояние $d=1$. В системе центра масс радиусы круговых орбит равны $m_2d/(m_1+m_2)$ и $m_1d/(m_1+m_2)$, а угловая скорость
$$\omega=\sqrt{\frac{G(m_1+m_2)}{d^3}}.$$
Точное положение в момент $t$ получается поворотом начальных радиус-векторов на угол $\omega t$. Разные массы позволяют проверить как схему интегрирования, так и множители $m_j$ в выражении для ускорения.

За один период $T=2\pi/\omega$ измерим максимальную ошибку положения по всем телам и узлам:
$$\varepsilon_r(h)=\max_{n,i}\|\mathbf r_{i,h}(t_n)-\mathbf r_{i,\mathrm{exact}}(t_n)\|.$$
Если $\varepsilon_r(h)\approx Ch^p$, то при делении шага пополам ошибка уменьшается в $2^p$ раз. Поэтому наблюдаемый порядок вычисляем как
$$p=\log_2\frac{\varepsilon_r(h)}{\varepsilon_r(h/2)}.$$
Отдельно проверяем, что для несимметричной тройки сумма внутренних сил и их суммарный момент равны нулю с точностью округления.

In [ ]:
def position_error(r, reference):
    # Ошибка во времени: в каждом узле выбираем наиболее неточное тело.
    return np.linalg.norm(r - reference, axis=2).max(axis=1)


test_m = np.array([1.0, 0.2])
test_r0 = np.array([[-test_m[1]/test_m.sum(), 0.0],
                    [ test_m[0]/test_m.sum(), 0.0]])
omega = np.sqrt(G*test_m.sum())
test_v0 = omega*np.column_stack((-test_r0[:, 1], test_r0[:, 0]))
test_period = 2*np.pi/omega
test_rows = []
for method in METHODS:
    previous = None
    for N in [100, 200, 400, 800]:
        sol = integrate(test_r0, test_v0, test_m, test_period, N, method)
        angle = omega*sol['t']
        exact = np.empty_like(sol['r'])
        exact[:, :, 0] = np.cos(angle)[:, None]*test_r0[:, 0]
        exact[:, :, 1] = np.sin(angle)[:, None]*test_r0[:, 0]
        error = position_error(sol['r'], exact).max()
        p = None if previous is None else np.log2(previous/error)
        test_rows.append([method, N, f'{error:.3e}', '—' if p is None else f'{p:.3f}'])
        previous = error
    expected = 4 if method == 'RK4' else 2
    assert abs(p - expected) < 0.15

table(['Метод', 'N', 'Максимальная ошибка положения', 'p'], test_rows)

# Внутренние силы в несимметричной тройке должны взаимно компенсироваться.
check_m = np.array([1.0, 0.7, 0.2])
check_r = np.array([[-1.0, 0.2], [0.4, -0.3], [1.2, 0.8]])
check_force = check_m[:, None]*acceleration(check_r, check_m)
np.testing.assert_allclose(check_force.sum(axis=0), 0.0, atol=2e-15)
np.testing.assert_allclose(
    np.sum(check_r[:, 0]*check_force[:, 1] - check_r[:, 1]*check_force[:, 0]),
    0.0, atol=2e-15,
)

На последних двух сетках наблюдаемый порядок равен 4,049 для RK4 и 2,000 для метода Верлета. Оба результата согласуются с теоретическими порядками схем.

## 2. Система центра масс и оценка погрешности

Начальные координаты и скорости переносим в систему центра масс:
$$
\mathbf r_i\leftarrow\mathbf r_i-\frac{\sum_jm_j\mathbf r_j}{\sum_jm_j},\qquad
\mathbf v_i\leftarrow\mathbf v_i-\frac{\sum_jm_j\mathbf v_j}{\sum_jm_j}.
$$
Сумма внутренних сил равна нулю, поэтому полный импульс сохраняется и центр масс движется равномерно. После такого перехода он должен оставаться в начале координат. В ходе расчёта координаты и скорости дополнительно не корректируются, поэтому возможный дрейф центра масс виден в численном решении.

Полная энергия и $z$-компонента момента импульса равны
$$
E=\frac12\sum_i m_i|\mathbf v_i|^2-G\sum_{i<j}\frac{m_im_j}{d_{ij}},\qquad
L_z=\sum_i m_i(x_iv_{y,i}-y_iv_{x,i}),\qquad
d_{ij}=|\mathbf r_i-\mathbf r_j|.
$$
В потенциальной энергии каждая пара учитывается один раз. Энергия сохраняется, поскольку силы определяются не зависящим от времени потенциалом, а момент импульса — вследствие центрального характера попарных сил.

Для контроля интегрирования используем
$$
\varepsilon_E=\max_n\frac{|E(t_n)-E(0)|}{|E(0)|},\qquad
\Delta L_{\max}=\max_n|L_z(t_n)-L_z(0)|.
$$
Энергия во всех примерах ненулевая. Изменение момента измеряем в абсолютных единицах: у «восьмёрки» $L_z(0)=0$, поэтому делить на него нельзя.

Ошибку положения в задаче трёх тел оцениваем относительно опорного решения:
$$
\varepsilon_r(h)=\max_{n,i}\|\mathbf r_{i,h}(t_n)-\mathbf r_{i,\mathrm{ref}}(t_n)\|.
$$
Максимум по телам учитывает отклонение лёгкого тела наравне с отклонением массивного. Опорную траекторию получаем RK4 на мелкой сетке и сопоставляем с расчётом, шаг которого вдвое больше. Во всех трёх примерах разность этих двух решений составляет менее 1% от наименьшей измеряемой $\varepsilon_r$. Это численная проверка сходимости опорного решения; точное решение задачи трёх тел заранее неизвестно.

In [ ]:
def pair_distances(r):
    return np.column_stack([
        np.linalg.norm(r[:, i] - r[:, j], axis=1) for i, j in PAIRS
    ])


def total_energy(r, v, masses):
    kinetic = 0.5*np.sum(masses[None, :, None]*v**2, axis=(1, 2))
    distances = pair_distances(r)
    potential = sum(-G*masses[i]*masses[j]/distances[:, k]
                    for k, (i, j) in enumerate(PAIRS))
    return kinetic + potential


def angular_momentum(r, v, masses):
    return np.sum(masses[None, :]*(r[:, :, 0]*v[:, :, 1]
                                 - r[:, :, 1]*v[:, :, 0]), axis=1)


def reference_and_runs(r0, v0, masses, t_end, grids, reference_N):
    reference = integrate(r0, v0, masses, t_end, reference_N, 'RK4')
    check = integrate(r0, v0, masses, t_end, reference_N//2, 'RK4')
    reference_gap = position_error(check['r'], reference['r'][::2]).max()
    runs = {}
    rows = []
    for method in METHODS:
        previous = None
        for N in grids:
            assert reference_N % N == 0
            sol = integrate(r0, v0, masses, t_end, N, method)
            aligned = reference['r'][::reference_N//N]
            np.testing.assert_allclose(sol['t'], reference['t'][::reference_N//N])
            sol['dr'] = position_error(sol['r'], aligned)
            sol['E'] = total_energy(sol['r'], sol['v'], masses)
            sol['L'] = angular_momentum(sol['r'], sol['v'], masses)
            sol['d'] = pair_distances(sol['r'])
            sol['dE'] = (sol['E'] - sol['E'][0])/abs(sol['E'][0])
            error = sol['dr'].max()
            p = None if previous is None else np.log2(previous/error)
            rows.append([method, N, f'{sol["h"]:.5g}', f'{error:.3e}',
                         '—' if p is None else f'{p:.2f}',
                         f'{np.abs(sol["dE"]).max():.3e}',
                         f'{np.abs(sol["L"]-sol["L"][0]).max():.2e}'])
            runs[method, N] = sol
            previous = error
    table(['Метод', 'N', 'h', 'εr', 'p', 'εE', 'ΔLmax'], rows)
    smallest_error = min(sol['dr'].max() for sol in runs.values())
    print(f'Опорные сетки: N = {reference_N//2} и {reference_N}. '
          f'Максимальная разность положений: {reference_gap:.2e}.')
    print(f'Это {reference_gap/smallest_error:.2%} от наименьшей εr в таблице.')
    assert reference_gap < 0.01*smallest_error
    return reference, runs

Схема Верлета сохраняет полный момент импульса в точной арифметике. При обновлении скорости суммарный момент внутренних центральных сил равен нулю. При обновлении координат приращение $h\mathbf v_{i,n+1/2}$ параллельно скорости, поэтому его вклад в $\mathbf r_i\times m_i\mathbf v_i$ также равен нулю. Таким образом, оба обновления скорости и обновление координат сохраняют $L_z$. Отклонение в вычислениях связано с округлением. Точного сохранения энергии эта схема не обеспечивает.

## 3. Звезда и две планеты

Массы звезды и планет равны $1$, $10^{-3}$ и $2\cdot10^{-3}$. До перехода в систему центра масс звезда находится в начале координат, планеты — в точках $(1,0)$ и $(1.8,0)$. Скорости планет направлены по оси $y$ и равны круговым скоростям в поле неподвижной звезды единичной массы: $v=\sqrt{G/r}$.

Начальные скорости получены в приближении круговых орбит. В полной задаче планеты возмущают движение друг друга, а звезда движется под действием их притяжения. Интегрируются уравнения всех трёх тел.

Время расчёта равно трём приближённым периодам внешней планеты:
$$t_{\mathrm{end}}=3T_{\mathrm{out}},\qquad T_{\mathrm{out}}=2\pi\sqrt{1.8^3/G}.$$
Используем $N=600,1200,2400$; опорные сетки содержат $9600$ и $19200$ шагов.

In [ ]:
hier_m = np.array([1.0, 1e-3, 2e-3])
hier_r0 = np.array([[0.0, 0.0], [1.0, 0.0], [1.8, 0.0]])
hier_v0 = np.array([[0.0, 0.0], [0.0, 1.0], [0.0, 1/np.sqrt(1.8)]])
hier_r0, hier_v0 = to_center_of_mass(hier_r0, hier_v0, hier_m)
T_outer = 2*np.pi*np.sqrt(1.8**3/G)
hier_end = 3*T_outer
hier_grids = [600, 1200, 2400]
hier_ref, hier_runs = reference_and_runs(
    hier_r0, hier_v0, hier_m, hier_end, hier_grids, 19200
)

Для метода Верлета деление шага пополам уменьшает ошибку положения почти в четыре раза. Для RK4 наблюдаемые порядки равны 4,70 и 4,54: на этих сетках отношение ошибок ещё не приблизилось к предельному значению $2^4=16$.

При $N=2400$ максимальная ошибка положения составляет $2.4\cdot10^{-7}$ для RK4 и $5.4\cdot10^{-3}$ для метода Верлета. Относительная ошибка энергии метода Верлета при этом около $1.4\cdot10^{-6}$. Малое изменение энергии может сопровождаться заметным расхождением положений: одной проверки энергии для оценки траектории недостаточно.

Траектории показаны при $N=1200$ в системе центра масс. Синяя сплошная линия соответствует RK4, оранжевый пунктир — методу Верлета. Пунктир нанесён поверх сплошной линии. Внутренняя орбита принадлежит первой планете, внешняя — второй. Справа траектория звезды показана в увеличенном масштабе.

На масштабе орбит решения почти совпадают. Нижний график показывает их различие количественно: $\delta_i(t)=\|\mathbf r_{i,\mathrm{RK4}}(t)-\mathbf r_{i,\mathrm{Верлета}}(t)\|$ в одни и те же моменты времени. Это разность двух численных решений; ошибки относительно опорной траектории приведены в таблице выше.

In [ ]:
fig = plt.figure(figsize=(10.5, 7.8), layout='constrained')
grid = fig.add_gridspec(2, 2, height_ratios=[2.7, 1.2])
orbit_axes = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])]
difference_ax = fig.add_subplot(grid[1, :])
rk = hier_runs['RK4', 1200]
verlet = hier_runs['Метод Верлета', 1200]
body_names = ['звезда', 'планета 1', 'планета 2']

for ax, bodies in zip(orbit_axes, [range(3), [0]]):
    for i in bodies:
        ax.plot(rk['r'][:, i, 0], rk['r'][:, i, 1],
                color=METHOD_COLORS['RK4'], lw=2.8, zorder=2)
        path = verlet['r'][:, i]
        # Начинаем пунктир заново на каждом обороте планеты:
        # иначе штрихи разных оборотов заполняют промежутки друг друга.
        turns = np.array([], dtype=int)
        if i > 0:
            turns = np.flatnonzero(
                (path[:-1, 1] < 0) & (path[1:, 1] >= 0) & (path[1:, 0] > 0)
            ) + 1
        cuts = np.r_[0, turns, len(path) - 1]
        for start, stop in zip(cuts[:-1], cuts[1:]):
            ax.plot(path[start:stop + 1, 0], path[start:stop + 1, 1],
                    color=METHOD_COLORS['Метод Верлета'], lw=1.8,
                    ls=(0, (4, 4)), dash_capstyle='butt', zorder=3)
    ax.plot(0, 0, '+', color='black', ms=9, mew=1.4, zorder=5)
    ax.set(xlabel='x', ylabel='y', aspect='equal')
orbit_axes[0].set_title('Вся система')
orbit_axes[1].set_title('Траектория звезды — увеличение')
orbit_axes[1].legend(handles=[
    Line2D([], [], color=METHOD_COLORS['RK4'], lw=2.8, label='RK4'),
    Line2D([], [], color=METHOD_COLORS['Метод Верлета'], lw=1.8,
           ls=(0, (4, 4)), label='Метод Верлета'),
], loc='upper center', bbox_to_anchor=(.5, -.15), ncol=2, fontsize=9)

separation = np.linalg.norm(rk['r'] - verlet['r'], axis=2)
for i, (color, marker, name, style) in enumerate(zip(
    BODY_COLORS, BODY_MARKERS, body_names, ['-', '--', '-.']
)):
    difference_ax.semilogy(rk['t'][1:]/T_outer, separation[1:, i],
                           color=color, ls=style, marker=marker, markevery=120,
                           ms=4, mfc='white', label=name)
difference_ax.set(xlabel='t / T внешней планеты', ylabel='δᵢ(t)',
                  title='Разность положений по двум методам', xlim=(0, 3))
difference_ax.legend(loc='lower right', ncol=3, fontsize=8)
fig.suptitle('Иерархическая система: N = 1200; крестик — центр масс')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, layout='constrained')
for col, method in enumerate(METHODS):
    sol = hier_runs[method, 1200]
    axes[0, col].plot(sol['t']/T_outer, sol['dE'], color=METHOD_COLORS[method])
    axes[1, col].plot(sol['t']/T_outer, sol['L']-sol['L'][0],
                      color=METHOD_COLORS[method])
    axes[0, col].set(title=method, ylabel='(E − E₀) / |E₀|')
    axes[1, col].set(xlabel='t / T внешней планеты', ylabel='Lz − Lz₀')
    for ax in axes[:, col]:
        ax.axhline(0, color='0.4', lw=.7)
        ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
fig.suptitle('Изменение энергии и момента; вертикальные масштабы различаются')
plt.show()

За три периода внешней планеты у RK4 наблюдается небольшой дрейф энергии и момента импульса. В расчёте методом Верлета энергия колеблется, а изменение момента остаётся на уровне округления. Вертикальные масштабы графиков различаются в соответствии с величинами ошибок.

## 4. Периодическая «восьмёрка»

Три равные массы могут двигаться по одной восьмёркообразной кривой со сдвигом на треть периода. Начальные данные взяты из подписи к рисунку 1 работы [A. Chenciner, R. Montgomery (2000)](https://www.maths.tcd.ie/EMIS/journals/Annals/152_3/chencine.pdf) с перестановкой номеров первых двух тел:
$$
\begin{array}{c|rr|rr}
i & x_i & y_i & v_{x,i} & v_{y,i} \\\hline
1&-0.97000436& 0.24308753& 0.466203685& 0.43236573\\
2& 0.97000436&-0.24308753& 0.466203685& 0.43236573\\
3&0&0&-0.93240737&-0.86473146
\end{array}
$$
Все массы равны 1, центр масс неподвижен, полный момент импульса равен нулю. Приближённый период из того же источника — $T_8=6.32591398$.

Наряду с ошибкой траектории относительно опорного решения измерим невязку возвращения:
$$R(h)=\max_i\|\mathbf r_{i,h}(T_8)-\mathbf r_i(0)\|.$$
Начальные данные и период округлены. Поэтому $R(h)$ включает как погрешность интегрирования, так и отклонение от точной периодичности, вызванное неточным заданием этих величин. Сходимость схемы проверяем по $\varepsilon_r(h)$ на всём интервале $[0,T_8]$.

In [ ]:
eight_m = np.ones(3)
eight_r0 = np.array([[-0.97000436, 0.24308753],
                     [0.97000436, -0.24308753], [0.0, 0.0]])
eight_v0 = np.array([[0.466203685, 0.43236573],
                     [0.466203685, 0.43236573], [-0.93240737, -0.86473146]])
T_eight = 6.32591398
eight_grids = [100, 200, 400, 800, 1600]
eight_ref, eight_runs = reference_and_runs(
    eight_r0, eight_v0, eight_m, T_eight, eight_grids, 25600
)

return_rows = []
for N in eight_grids:
    row = [N]
    for method in METHODS:
        sol = eight_runs[method, N]
        sol['return_error'] = np.linalg.norm(sol['r'][-1]-eight_r0, axis=1).max()
        row.append(f'{sol["return_error"]:.3e}')
    return_rows.append(row)
table(['N', 'R: RK4', 'R: Метод Верлета'], return_rows)
eight_return_ref = np.linalg.norm(eight_ref['r'][-1]-eight_r0, axis=1).max()
print(f'Невязка возвращения опорного решения: {eight_return_ref:.3e}.')

Относительно опорного решения RK4 сходится с порядком около 4, а метод Верлета — около 2. Невязка возвращения RK4 при переходе от $N=800$ к $N=1600$, напротив, возрастает с $2.2\cdot10^{-8}$ до $4.0\cdot10^{-8}$. Это объясняется разложением
$$
\mathbf r_{i,h}(T_8)-\mathbf r_i(0)
=\bigl[\mathbf r_{i,h}(T_8)-\mathbf r_{i,\mathrm{ref}}(T_8)\bigr]
+\bigl[\mathbf r_{i,\mathrm{ref}}(T_8)-\mathbf r_i(0)\bigr].
$$
Первое слагаемое уменьшается при дроблении шага. Второе задаёт невязку опорного решения, около $4.1\cdot10^{-8}$, которая сохраняется из-за округления начальных данных и периода. При $N=800$ эти векторные отклонения частично компенсируются; при $N=1600$ результат приближается к невязке опорного решения. Порядок метода поэтому определяется по ошибке траектории, а не по одному расстоянию до начальной точки.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), layout='constrained')
for method in METHODS:
    color = METHOD_COLORS[method]
    errors = [eight_runs[method, N]['dr'].max() for N in eight_grids]
    returns = [eight_runs[method, N]['return_error'] for N in eight_grids]
    axes[0].loglog(eight_grids, errors, color=color, ls=METHOD_LINES[method],
                   marker=METHOD_MARKERS[method], ms=6, mfc='white', mew=1.4, label=method)
    axes[1].loglog(eight_grids, returns, color=color, ls=METHOD_LINES[method],
                   marker=METHOD_MARKERS[method], ms=6, mfc='white', mew=1.4, label=method)
axes[1].axhline(eight_return_ref, color='0.35', ls=':', label='возврат опорного решения')
axes[0].set(title='Сходимость численной траектории', ylabel='εr — ошибка относительно опоры')
axes[1].set(title='Невязка возвращения при t = T₈', ylabel='R — расстояние до начальной точки')
for ax in axes:
    ax.set_xlabel('Число шагов N за период')
    ax.set_xticks(eight_grids, [str(N) for N in eight_grids])
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.legend(fontsize=8)
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.8), layout='constrained')
sol = eight_runs['RK4', 800]
frame = 200  # четверть периода
ax.plot(sol['r'][:, 0, 0], sol['r'][:, 0, 1], color='0.65', lw=1, zorder=1)
for i, color in enumerate(BODY_COLORS):
    trail = sol['r'][max(0, frame-120):frame+1, i]
    ax.plot(trail[:, 0], trail[:, 1], color=color, lw=2)
    ax.plot(*sol['r'][frame, i], ls='none', marker=BODY_MARKERS[i], color=color,
            ms=8, mec='white', mew=.9, zorder=4, label=f'тело {i+1}')
ax.set(aspect='equal', xlabel='x', ylabel='y', title='«Восьмёрка»: RK4, t = T₈/4, N = 800')
ax.legend(ncol=3, fontsize=9, loc='upper center', bbox_to_anchor=(.5, -.22))
plt.show()

### Анимация «восьмёрки»

Результаты RK4 и метода Верлета показаны при одинаковом шаге $h=T_8/800$ с общей шкалой времени и одинаковым масштабом координат. Серый контур — опорная орбита, цветные следы — участки пути трёх тел за последние $0.15T_8$. Тела поочерёдно проходят обе петли и центр восьмёрки; пересечение орбиты в начале координат не означает столкновения, поскольку тела проходят его в разные моменты.

In [ ]:
def show_animation(animation):
    html = animation.to_jshtml(default_mode='once')
    # Заменяем внешний шрифт кнопок обычными символами для просмотра без сети.
    html = re.sub(r'<link\s+rel="stylesheet"\s+href="[^"]+">', '', html)
    icons = {'fa-play': '▶', 'fa-play fa-flip-horizontal': '◀', 'fa-pause': 'Ⅱ',
             'fa-step-forward': '▶|', 'fa-step-backward': '|◀',
             'fa-fast-forward': '⏭', 'fa-fast-backward': '⏮',
             'fa-minus': '−', 'fa-plus': '+'}
    html = re.sub(r'<i class="fa ([^"]+)"[^>]*></i>',
                  lambda match: icons.get(match[1], ''), html)
    display(HTML(html))


def animate_eight():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.9), dpi=90)
    artists = []
    for ax, method in zip(axes, METHODS):
        ax.plot(eight_ref['r'][:, 0, 0], eight_ref['r'][:, 0, 1],
                color='0.65', lw=1, zorder=1)
        trails, points = [], []
        for i, color in enumerate(BODY_COLORS):
            trails.append(ax.plot([], [], color=color, lw=2, zorder=2)[0])
            points.append(ax.plot([], [], ls='none', marker=BODY_MARKERS[i],
                                  color=color, ms=8, mec='white', mew=.9, zorder=4)[0])
        ax.set(xlim=(-1.15, 1.15), ylim=(-.43, .43), aspect='equal',
               xlabel='x', ylabel='y', title=method)
        artists.append((trails, points))
    fig.legend(handles=[Line2D([], [], color=c, marker=BODY_MARKERS[i], label=f'тело {i+1}')
                        for i, c in enumerate(BODY_COLORS)],
               loc='lower center', ncol=3, frameon=False)
    title = fig.suptitle('')
    fig.subplots_adjust(left=.075, right=.98, wspace=.28, bottom=.22, top=.8)

    def update(k):
        for method, (trails, points) in zip(METHODS, artists):
            r = eight_runs[method, 800]['r']
            for i, (trail, point) in enumerate(zip(trails, points)):
                trail.set_data(r[max(0, k-120):k+1, i, 0], r[max(0, k-120):k+1, i, 1])
                point.set_data([r[k, i, 0]], [r[k, i, 1]])
        title.set_text(f'Периодическая восьмёрка: t / T₈ = {k/800:.3f}')

    animation = FuncAnimation(fig, update, frames=range(0, 801, 5),
                              interval=50, repeat=False)
    plt.close(fig)
    return animation


eight_animation = animate_eight()
show_animation(eight_animation)

## 5. Пролёт третьего тела около связанной пары

Зададим круговую двойную систему с $m_1=m_2=1$, $\mathbf r_1=(-1/2,0)$ и $\mathbf r_2=(1/2,0)$. Из равенства центростремительного ускорения ускорению гравитационного притяжения
$$\frac{v^2}{1/2}=\frac{G}{1^2}$$
получаем модуль скорости каждого тела $v=1/\sqrt2$. Первое тело движется вниз, второе — вверх. Третье тело массы $m_3=0.1$ начинает движение в точке $(-4,0)$ со скоростью $(1,0)$. Все данные переносим в систему центра масс тройки.

Круговые скорости соответствуют изолированной паре. Притяжение третьего тела возмущает её движение, причём сильнее всего — во время близкого пролёта. Это позволяет сопоставить изменение самой пары с ошибками интегрирования на участке сближения.

На интервале $0\le t\le8$ используем шаги $0.02$, $0.01$, $0.005$, $0.0025$ и $0.00125$. Опорное решение получено RK4 с $h=0.000125$ и проверено сравнением с $h=0.00025$.

In [ ]:
fly_m = np.array([1.0, 1.0, 0.1])
fly_r0 = np.array([[-.5, 0.0], [.5, 0.0], [-4.0, 0.0]])
fly_v0 = np.array([[0.0, -1/np.sqrt(2)], [0.0, 1/np.sqrt(2)], [1.0, 0.0]])
fly_r0, fly_v0 = to_center_of_mass(fly_r0, fly_v0, fly_m)
fly_end = 8.0
fly_grids = [400, 800, 1600, 3200, 6400]
fly_ref, fly_runs = reference_and_runs(
    fly_r0, fly_v0, fly_m, fly_end, fly_grids, 64000
)

fly_distances = pair_distances(fly_ref['r'])
encounter_index, pair_index = np.unravel_index(fly_distances.argmin(), fly_distances.shape)
encounter_time = fly_ref['t'][encounter_index]
encounter_distance = fly_distances[encounter_index, pair_index]
i, j = PAIRS[pair_index]
print(f'Ближайшее сближение: тела {i+1} и {j+1}, '
      f'd ≈ {encounter_distance:.4f}, t ≈ {encounter_time:.4f}.')

При уменьшении шага с $0.02$ до $0.005$ ошибка положения RK4 снижается с $0.114$ до $1.86\cdot10^{-4}$. Для метода Верлета на тех же сетках эта ошибка равна $0.388$ и $0.0250$. На последних сетках наблюдаемый порядок составляет 4,21 для RK4 и 2,00 для метода Верлета.

Выберем допуски $\varepsilon_r<0.005$ и $\varepsilon_E<10^{-3}$. Первый ограничивает отклонение любого тела половиной процента от начального расстояния в паре, второй — изменение полной энергии одной тысячной модуля начальной энергии. По этим двум условиям найдём наибольший допустимый шаг среди проверенных сеток.

In [ ]:
position_tolerance = 0.005
energy_tolerance = 1e-3
accepted_rows = []
for method in METHODS:
    accepted = [fly_runs[method, N] for N in fly_grids
                if fly_runs[method, N]['dr'].max() < position_tolerance
                and np.abs(fly_runs[method, N]['dE']).max() < energy_tolerance]
    if accepted:
        sol = max(accepted, key=lambda s: s['h'])
        accepted_rows.append([method, f'{sol["h"]:g}',
                              f'{sol["dr"].max():.3e}',
                              f'{np.abs(sol["dE"]).max():.3e}'])
    else:
        accepted_rows.append([method, 'подходящего шага нет', '—', '—'])
table(['Метод', 'Наибольший подходящий h', 'εr', 'εE'], accepted_rows)

Среди проверенных шагов оба требования к точности выполняются при $h\le0.01$ для RK4 и при $h=0.00125$ для метода Верлета. Такую точность требуется обеспечить на всём интервале $0\le t\le8$, включая короткое тесное сближение.

Верхний ряд следующего рисунка показывает минимальное попарное расстояние в опорном решении. Численные кривые на масштабе этого расстояния почти совпадают, поэтому в среднем ряду показаны их отклонения $d_{\min,h}(t)-d_{\min,\mathrm{ref}}(t)$. Нижний ряд — относительная ошибка энергии. Во всех трёх рядах время согласовано: слева показан весь расчёт, справа — окрестность сближения.

Каждому сочетанию метода и шага соответствует свой цвет; штриховые линии обозначают $h=0.02$, сплошные — $h=0.00125$. Та же легенда используется на следующем графике ошибки положения. Вертикальная линия отмечает минимум опорного расстояния. Для знакопеременной разности расстояний шкала линейна в диапазоне $\pm10^{-7}$ и логарифмическая вне его.

In [ ]:
run_styles = [
    ('RK4', 400, '#0072B2', (0, (6, 3)), 'o'),
    ('RK4', 6400, '#332288', '-', 'o'),
    ('Метод Верлета', 400, '#D55E00', (0, (6, 3)), 's'),
    ('Метод Верлета', 6400, '#008568', '-', 's'),
]
reference_minimum = fly_distances.min(axis=1)
run_handles = [Line2D([], [], color=color, ls=style, marker=marker,
                       ms=5, mfc='white', label=f'{method}, h = {fly_runs[method, N]["h"]:g}')
               for method, N, color, style, marker in run_styles]

fig, axes = plt.subplots(3, 2, figsize=(11, 9.1), sharex='col', sharey='row')
for col in range(2):
    axes[0, col].plot(fly_ref['t'], reference_minimum, color='0.25', lw=1.9)
    axes[0, col].plot(encounter_time, encounter_distance, 'o', color='black', ms=4)
    for run_index, (method, N, color, style, marker) in enumerate(run_styles):
        sol = fly_runs[method, N]
        ref_on_grid = reference_minimum[::fly_ref['N']//N]
        difference = sol['d'].min(axis=1)-ref_on_grid
        # Несколько маркеров на каждой кривой дублируют цвет и тип линии.
        spacing = max(1, int((.18 if col else .9)/sol['h']))
        mark = (run_index*spacing//4, spacing)
        axes[1, col].plot(sol['t'], difference, color=color, ls=style, lw=1.6,
                          marker=marker, markevery=mark, ms=4, mfc='white', mew=1)
        axes[2, col].semilogy(sol['t'], np.maximum(np.abs(sol['dE']), 1e-16),
                              color=color, ls=style, lw=1.7, marker=marker,
                              markevery=mark, ms=4, mfc='white', mew=1)
    for ax in axes[:, col]:
        ax.axvline(encounter_time, color='0.35', ls=':', lw=1)
    axes[1, col].set_yscale('symlog', linthresh=1e-7, linscale=.6)
    axes[1, col].set_yticks([-1e-2, -1e-4, -1e-6, 0, 1e-6, 1e-4, 1e-2])
    axes[1, col].axhline(0, color='0.55', lw=.6)
    axes[2, col].set(xlabel='t', ylim=(1e-13, .15))
axes[0, 0].set(title='Весь интервал', ylabel='min dᵢⱼ — опорное решение', xlim=(0, 8))
axes[0, 1].set(title='Окрестность близкого сближения', xlim=(2.1, 2.85))
axes[1, 0].set_ylabel('dmin − dmin, ref')
axes[2, 0].set_ylabel('|E − E₀| / |E₀|')
fig.suptitle('Пролёт: расстояние между телами и численная ошибка', y=.99)
fig.legend(handles=run_handles, loc='lower center', ncol=2, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, .07, 1, .975))
plt.show()

fig, ax = plt.subplots(figsize=(9, 4.3))
for run_index, (method, N, color, style, marker) in enumerate(run_styles):
    sol = fly_runs[method, N]
    spacing = max(1, int(.9/sol['h']))
    ax.semilogy(sol['t'][1:], sol['dr'][1:], color=color, ls=style, lw=1.8,
                marker=marker, markevery=(run_index*spacing//4, spacing),
                ms=4, mfc='white', mew=1)
ax.axvline(encounter_time, color='0.35', ls=':')
ax.set(xlabel='t', ylabel='Максимальная ошибка положения тел',
       title='Ошибка траектории до и после пролёта', xlim=(0, 8))
fig.legend(handles=run_handles, loc='lower center', ncol=2, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, .15, 1, 1))
plt.show()

Резкий рост ошибки энергии приходится на сближение тел 2 и 3 около $t=2.484$, когда расстояние между ними уменьшается до $0.1146$. На мелкой сетке скачок значительно слабее. После пролёта ошибка положения продолжает расти: отклонение скорости, накопленное при сближении, влияет на последующее движение.

Изменение исходной пары оценим по относительным координатам и скоростям. Её энергия на единицу приведённой массы и удельный момент относительного движения равны
$$
\epsilon_{12}=\frac{|\mathbf v_2-\mathbf v_1|^2}{2}-\frac{G(m_1+m_2)}{|\mathbf r_2-\mathbf r_1|},
$$
$$
\ell_{12}=(x_2-x_1)(v_{y,2}-v_{y,1})-(y_2-y_1)(v_{x,2}-v_{x,1}).
$$
Если в данный момент исключить действие третьего тела, дальнейшая двухтельная орбита при $\epsilon_{12}<0$ имела бы большую полуось и эксцентриситет
$$
a_{12}=-\frac{G(m_1+m_2)}{2\epsilon_{12}},\qquad
e_{12}=\sqrt{1+\frac{2\epsilon_{12}\ell_{12}^2}{G^2(m_1+m_2)^2}}.
$$
В полной задаче эти параметры меняются вследствие обмена энергией и моментом импульса с третьим телом. Вычислим их по опорному решению до пролёта и при $t=8$.

In [ ]:
pair_rows = []
for k in [0, -1]:
    dr = fly_ref['r'][k, 1]-fly_ref['r'][k, 0]
    dv = fly_ref['v'][k, 1]-fly_ref['v'][k, 0]
    mu_pair = G*(fly_m[0]+fly_m[1])
    eps = np.dot(dv, dv)/2 - mu_pair/np.linalg.norm(dr)
    ell = dr[0]*dv[1] - dr[1]*dv[0]
    a_pair = -mu_pair/(2*eps)
    e_pair = np.sqrt(max(0.0, 1 + 2*eps*ell**2/mu_pair**2))
    pair_rows.append([f'{fly_ref["t"][k]:g}', f'{eps:.6f}', f'{a_pair:.4f}', f'{e_pair:.4f}'])
table(['t', 'ε₁₂', 'a₁₂', 'e₁₂'], pair_rows)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), layout='constrained')
for i, color in enumerate(BODY_COLORS):
    r = fly_ref['r'][:, i]
    axes[0].plot(r[:, 0], r[:, 1], color=color, lw=1.8, label=f'тело {i+1}')
    axes[0].plot(*r[0], ls='none', marker=BODY_MARKERS[i], mfc='white', mec=color, ms=7, mew=1.3, zorder=4)
    axes[0].plot(*r[-1], ls='none', marker=BODY_MARKERS[i], color=color, ms=7, mec='white', mew=.7, zorder=4)
for k, (i, j) in enumerate(PAIRS):
    axes[1].plot(fly_ref['t'], fly_distances[:, k], color=BODY_COLORS[k],
                  ls=['-', '--', '-.'][k], marker=BODY_MARKERS[k], markevery=6400,
                  ms=4, mfc='white', label=f'd{i+1}{j+1}')
axes[0].set(xlabel='x', ylabel='y', aspect='equal', title='Траектории в системе центра масс')
axes[0].legend(fontsize=9)
axes[1].axvline(encounter_time, color='0.35', ls=':')
axes[1].set(xlabel='t', ylabel='Расстояние', title='Исходная пара остаётся связанной')
axes[1].legend(fontsize=9)
plt.show()

При $t=8$ энергия относительного движения пары отрицательна. Её мгновенная большая полуось уменьшилась с 1 до 0,906, а эксцентриситет вырос с 0 до 0,273. Пролёт не разрушил пару, но сделал её движение заметно некруговым. Расстояния от третьего тела до обоих компонентов пары в конце расчёта растут. На траекториях полые маркеры отмечают начальные положения, закрашенные — положения при $t=8$.

### Анимация пролёта

Показано опорное решение RK4 с $h=0.000125$ в системе центра масс. На левой панели показана вся тройка, на правой — окрестность исходной пары в увеличенном масштабе. Цветные следы показывают отклонение каждого тела при взаимодействии; отрезок соединяет ближайшие тела. Указатель на графике расстояния отмечает текущий момент движения.

In [ ]:
def animate_flyby():
    fig = plt.figure(figsize=(9, 6.2), dpi=90)
    grid = fig.add_gridspec(2, 2, height_ratios=[3.8, 1], width_ratios=[.85, 1.2],
                           left=.09, right=.98, bottom=.09, top=.87, hspace=.4, wspace=.28)
    overview = fig.add_subplot(grid[0, 0])
    close = fig.add_subplot(grid[0, 1])
    distance_ax = fig.add_subplot(grid[1, :])
    panels = []
    for ax in [overview, close]:
        trails, points = [], []
        for i, color in enumerate(BODY_COLORS):
            trails.append(ax.plot([], [], color=color, lw=1.8, zorder=2)[0])
            points.append(ax.plot([], [], ls='none', marker=BODY_MARKERS[i],
                                  color=color, ms=9 if i<2 else 8,
                                  mec='white', mew=.9, zorder=5)[0])
        connector = ax.plot([], [], ':', color='0.2', lw=1.2, zorder=3)[0]
        ax.set(xlabel='x', ylabel='y', aspect='equal')
        panels.append((trails, points, connector))
    overview.set(xlim=(-4.3, 2.0), ylim=(-8.7, 1.0), title='Вся система')
    close.set(xlim=(-1.35, 1.35), ylim=(-1.15, 1.25), title='Окрестность пары')
    min_distance = fly_distances.min(axis=1)
    distance_ax.plot(fly_ref['t'], min_distance, color='0.3', lw=1)
    distance_ax.axvline(encounter_time, color='0.65', ls=':', lw=1)
    cursor = distance_ax.axvline(0, color='#9B1762', lw=1.8)
    dot = distance_ax.plot([], [], 'o', color='#9B1762', ms=6)[0]
    distance_ax.set(xlim=(0, 8), ylim=(0, 1.25), xlabel='t', ylabel='min dᵢⱼ')
    title = fig.suptitle('')
    fig.legend(handles=[Line2D([], [], color=c, marker=BODY_MARKERS[i], label=f'тело {i+1}')
                        for i, c in enumerate(BODY_COLORS)], loc='upper center',
               bbox_to_anchor=(.5, .945), ncol=3, frameon=False, fontsize=9)

    def update(k):
        a, b = PAIRS[np.argmin(fly_distances[k])]
        for trails, points, connector in panels:
            for i, (trail, point) in enumerate(zip(trails, points)):
                trail.set_data(fly_ref['r'][:k+1, i, 0], fly_ref['r'][:k+1, i, 1])
                point.set_data([fly_ref['r'][k, i, 0]], [fly_ref['r'][k, i, 1]])
            connector.set_data(fly_ref['r'][k, [a, b], 0], fly_ref['r'][k, [a, b], 1])
        t = fly_ref['t'][k]
        cursor.set_xdata([t, t])
        dot.set_data([t], [min_distance[k]])
        title.set_text(f'Пролёт: t = {t:.3f}; ближайшие тела {a+1}–{b+1}, d = {min_distance[k]:.3f}')

    animation = FuncAnimation(fig, update, frames=range(0, 64001, 200),
                              interval=35, repeat=False)
    plt.close(fig)
    return animation


fly_animation = animate_flyby()
show_animation(fly_animation)

## 6. Требования к шагу в разных участках движения

При тесном сближении ускорение возрастает, а его направление быстро меняется. Для каждой пары оценим два местных масштаба времени:
$$
\tau_{g,ij}=\sqrt{\frac{d_{ij}^3}{G(m_i+m_j)}},\qquad
\tau_{v,ij}=\frac{d_{ij}}{|\mathbf v_i-\mathbf v_j|}.
$$
Первый следует из относительного ускорения $a\sim G(m_i+m_j)/d^2$ и оценки перемещения $d\sim a\tau_g^2$. Второй — время прохождения расстояния $d$ с текущей относительной скоростью. Для кругового движения пары эти масштабы совпадают; при быстром пролёте $\tau_v$ может быть меньше $\tau_g$.

Чтобы разрешить локальное изменение движения, нужен шаг $h\ll\min(\tau_g,\tau_v)$. Это оценка характерного масштаба; выполнение заданных требований к точности проверяется сравнением решений при уменьшении шага. Минимумы в таблице взяты по всем парам и всем узлам опорного расчёта.

In [ ]:
scale_rows = []
for label, ref, masses in [('звезда и планеты', hier_ref, hier_m),
                            ('восьмёрка', eight_ref, eight_m),
                            ('пролёт', fly_ref, fly_m)]:
    distances = pair_distances(ref['r'])
    relative_speeds = np.column_stack([
        np.linalg.norm(ref['v'][:, i]-ref['v'][:, j], axis=1) for i,j in PAIRS
    ])
    pair_masses = np.array([masses[i]+masses[j] for i,j in PAIRS])
    tau_g = np.sqrt(distances**3/(G*pair_masses[None, :]))
    tau_v = np.divide(distances, relative_speeds,
                      out=np.full_like(distances, np.inf), where=relative_speeds>0)
    fastest = np.minimum(tau_g, tau_v)
    k, p = np.unravel_index(fastest.argmin(), fastest.shape)
    i, j = PAIRS[p]
    scale_rows.append([label, f'{distances.min():.4f}', f'{tau_g.min():.4f}',
                       f'{tau_v.min():.4f}', f'{i+1}–{j+1}', f'{ref["t"][k]:.3f}'])
table(['Движение', 'min dᵢⱼ', 'min τg', 'min τv',
       'Пара при min(τg, τv)', 't'], scale_rows)

# Проверяем отсутствие заметного дрейфа системы отсчёта во всех основных расчётах.
frame_rows = []
for label, runs, masses in [('звезда и планеты', hier_runs, hier_m),
                             ('восьмёрка', eight_runs, eight_m),
                             ('пролёт', fly_runs, fly_m)]:
    cm_error, momentum_error = 0.0, 0.0
    for sol in runs.values():
        cm = np.sum(masses[None, :, None]*sol['r'], axis=1)/masses.sum()
        momentum = np.sum(masses[None, :, None]*sol['v'], axis=1)
        cm_error = max(cm_error, np.linalg.norm(cm, axis=1).max())
        momentum_error = max(momentum_error, np.linalg.norm(momentum, axis=1).max())
        assert np.all(np.isfinite(sol['r'])) and np.all(sol['d'] > 0)
    frame_rows.append([label, f'{cm_error:.2e}', f'{momentum_error:.2e}'])
table(['Движение', 'max |r центра масс|', 'max |полного импульса|'], frame_rows)

В иерархической системе минимальное расстояние $0.798$ достигается между лёгкими планетами, однако самый короткий динамический масштаб времени, около $0.97$, задаётся движением внутренней планеты около звезды. Следовательно, ограничение на шаг зависит не только от расстояния, но и от масс и относительных скоростей.

У «восьмёрки» минимальное расстояние равно примерно $0.691$, а меньший из двух масштабов времени — $0.315$. Наиболее тесное взаимодействие возникает при пролёте: $d_{\min}\approx0.115$, $\tau_g\approx0.037$, $\tau_v\approx0.0255$. При $h=0.02$ на время $\tau_v$ приходится лишь 1,3 шага. Такая сетка слишком груба для точного прохождения сближения, что согласуется со скачком ошибки энергии на графике.

Расстояние от центра масс до начала координат во всех расчётах меньше $9\cdot10^{-15}$, а модуль полного импульса остаётся меньше $8\cdot10^{-15}$. Заметного численного дрейфа системы отсчёта нет.

## 7. Выводы

RK4 и метод Верлета воспроизводят все три режима и сходятся при уменьшении шага. Проверка на круговом двухтельном движении подтверждает четвёртый и второй порядки соответственно. При одинаковом шаге RK4 в рассмотренных примерах точнее по координатам и энергии; метод Верлета сохраняет полный момент импульса до округления.

Для периодической «восьмёрки» ошибка интегрирования и невязка возвращения различаются. Последняя достигает уровня, заданного округлением начальных данных и периода, поэтому по ней одной нельзя судить о сходимости метода.

Наиболее строгие требования к шагу предъявляет близкий пролёт тел 2 и 3 около $t=2.484$. Ошибка на этом коротком участке определяет точность последующей траектории. Для допусков $\varepsilon_r<0.005$ и $\varepsilon_E<10^{-3}$ на интервале $[0,8]$ наибольшие подходящие шаги среди проверенных равны $0.01$ для RK4 и $0.00125$ для метода Верлета.

Условие: [README курса, задание 2.2](https://github.com/yartmihant/Students6Year2026/blob/94006394dda778773c997fed6ef2d2a28b626104/README.md). Начальные данные «восьмёрки»: A. Chenciner, R. Montgomery, *A remarkable periodic solution of the three-body problem in the case of equal masses*, Annals of Mathematics 152 (2000), 881–901, [рисунок 1](https://www.maths.tcd.ie/EMIS/journals/Annals/152_3/chencine.pdf).